# Stage 06: Data Preprocessing

**Project:** SPY Next-Day High-Volatility Risk Alert  
**Date:** 2026-08-23

This notebook completes the Stage 06 starter using its provided seven-row dataset. It applies the required reusable functions—`drop_missing`, `fill_missing_median`, and `normalize_data`—saves a cleaned dataset, and compares every transformation stage.

In [1]:
# Packages needed (run once in the Stage 02 environment if missing):
# %pip install pandas

## 1. Reproducible paths and imports

The notebook searches upward for `homework/homework06`, so it runs from either the repository root or its own directory. The raw file is course-provided input and is never overwritten; processed outputs are recreated deterministically.

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def locate_homework_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "homework06" and (candidate / "src").exists():
            return candidate
        nested = candidate / "homework" / "homework06"
        if (nested / "src").exists():
            return nested
    raise FileNotFoundError("Could not locate homework/homework06")


ROOT = locate_homework_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data

print("Homework root:", ROOT)
print("Cleaning module:", ROOT / "src" / "cleaning.py")

Homework root: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework06
Cleaning module: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework06/src/cleaning.py


## 2. Load and profile the raw dataset

`zipcode` and `city` are loaded as strings because they are identifiers/categories, not continuous numeric measurements. Blank numeric cells remain missing values. The literal category `Unknown` is retained as observed text.

In [3]:
raw_path = RAW / "sample_data.csv"
if not raw_path.exists():
    raise FileNotFoundError(f"Required course dataset is missing: {raw_path}")

df_original = pd.read_csv(
    raw_path,
    dtype={"zipcode": "string", "city": "string"},
)
print("Raw path:", raw_path.relative_to(ROOT))
print("Shape:", df_original.shape)
display(df_original)
display(df_original.dtypes.rename("dtype").to_frame())

Raw path: data/raw/sample_data.csv
Shape: (7, 6)


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


,dtype
age,float64
income,float64
score,float64
zipcode,string
city,string
extra_data,float64


In [4]:
missing_profile = (
    pd.DataFrame(
        {
            "missing_count": df_original.isna().sum(),
            "missing_fraction": df_original.isna().mean(),
        }
    )
    .assign(missing_percent=lambda table: (100 * table["missing_fraction"]).round(1))
    .sort_values("missing_fraction", ascending=False)
)
display(missing_profile)

,missing_count,missing_fraction,missing_percent
extra_data,5,0.714286,71.4
income,3,0.428571,42.9
age,1,0.142857,14.3
score,1,0.142857,14.3
zipcode,0,0.000000,0.0
city,0,0.000000,0.0


## 3. Drop features with excessive missingness

Policy: remove columns whose missing fraction is **greater than** 50%. `extra_data` is 71.4% missing and has only two observations, so imputing it would manufacture most of the column. Rows are retained because the remaining numeric gaps can be demonstrated with median imputation.

In [5]:
df_after_drop = drop_missing(df_original, threshold=0.5, axis="columns")
dropped_columns = sorted(set(df_original.columns) - set(df_after_drop.columns))
print("Dropped columns:", dropped_columns)
print("Shape after drop:", df_after_drop.shape)
display(df_after_drop)

Dropped columns: ['extra_data']
Shape after drop: (7, 5)


,age,income,score,zipcode,city
0,34.0,55000.0,0.82,90210,Beverly
1,45.0,NaN,0.91,10001,New York
2,29.0,42000.0,NaN,60614,Chicago
3,50.0,58000.0,0.76,94103,SF
4,38.0,NaN,0.88,73301,Austin
5,NaN,NaN,0.65,12345,Unknown
6,41.0,49000.0,0.79,94105,San Francisco


## 4. Fill numeric gaps with medians

Policy: impute `age`, `income`, and `score` with their observed medians. Median imputation is transparent and less sensitive to extreme values than mean imputation, but it reduces variance and ignores relationships among features.

In [6]:
numeric_columns = ["age", "income", "score"]
imputation_medians = df_after_drop[numeric_columns].median().rename("median_used")
df_after_imputation = fill_missing_median(df_after_drop, numeric_columns)

display(imputation_medians.to_frame())
print("Missing cells after imputation:", int(df_after_imputation.isna().sum().sum()))
display(df_after_imputation)

,median_used
age,39.500
income,52000.000
score,0.805


Missing cells after imputation: 0


,age,income,score,zipcode,city
0,34.0,55000.0,0.820,90210,Beverly
1,45.0,52000.0,0.910,10001,New York
2,29.0,42000.0,0.805,60614,Chicago
3,50.0,58000.0,0.760,94103,SF
4,38.0,52000.0,0.880,73301,Austin
5,39.5,52000.0,0.650,12345,Unknown
6,41.0,49000.0,0.790,94105,San Francisco


## 5. Min-max normalize numeric features

Policy: replace the three numeric columns with min-max values in `[0, 1]`. This makes their scales comparable while preserving ordering. `zipcode` and `city` remain unchanged. In a real model, the minimum, maximum, and imputation medians must be fitted on training data only.

In [7]:
range_before_scaling = df_after_imputation[numeric_columns].agg(["min", "max"]).T
df_cleaned = normalize_data(
    df_after_imputation,
    numeric_columns,
    method="minmax",
)
range_after_scaling = df_cleaned[numeric_columns].agg(["min", "max"]).T

range_comparison = range_before_scaling.join(
    range_after_scaling,
    lsuffix="_before",
    rsuffix="_after",
)
display(range_comparison)
display(df_cleaned)

,min_before,max_before,min_after,max_after
age,29.00,50.00,0.0,1.0
income,42000.00,58000.00,0.0,1.0
score,0.65,0.91,0.0,1.0


,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin
5,0.500000,0.6250,0.000000,12345,Unknown
6,0.571429,0.4375,0.538462,94105,San Francisco


## 6. Compare stages and save processed outputs

The audit table records how shape, missingness, and duplicates change. The cleaned dataset and audit summary use stable paths so rerunning the notebook reproduces the same deliverables.

In [8]:
def stage_summary(stage: str, frame: pd.DataFrame) -> dict[str, object]:
    return {
        "stage": stage,
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "duplicate_rows": int(frame.duplicated().sum()),
    }


cleaning_summary = pd.DataFrame(
    [
        stage_summary("original", df_original),
        stage_summary("after_drop_missing", df_after_drop),
        stage_summary("after_median_imputation", df_after_imputation),
        stage_summary("after_minmax_normalization", df_cleaned),
    ]
)
display(cleaning_summary)

,stage,rows,columns,missing_cells,duplicate_rows
0,original,7,6,10,0
1,after_drop_missing,7,5,5,0
2,after_median_imputation,7,5,0,0
3,after_minmax_normalization,7,5,0,0


In [9]:
cleaned_path = PROCESSED / "sample_data_cleaned.csv"
summary_path = PROCESSED / "cleaning_summary.csv"
df_cleaned.to_csv(cleaned_path, index=False)
cleaning_summary.to_csv(summary_path, index=False)
print("Saved cleaned dataset:", cleaned_path.relative_to(ROOT))
print("Saved audit summary:", summary_path.relative_to(ROOT))

Saved cleaned dataset: data/processed/sample_data_cleaned.csv
Saved audit summary: data/processed/cleaning_summary.csv


## 7. Reload validation

The saved CSV is reloaded with explicit string dtypes, then checked against the in-memory result. This verifies the processed artifact—not only the transformation code.

In [10]:
df_reloaded = pd.read_csv(
    cleaned_path,
    dtype={"zipcode": "string", "city": "string"},
)
pd.testing.assert_frame_equal(df_cleaned, df_reloaded)

validation_checks = {
    "expected_shape_7_by_5": df_reloaded.shape == (7, 5),
    "expected_columns": list(df_reloaded.columns)
    == ["age", "income", "score", "zipcode", "city"],
    "no_missing_values": not df_reloaded.isna().any().any(),
    "no_duplicate_rows": not df_reloaded.duplicated().any(),
    "numeric_min_is_zero": df_reloaded[numeric_columns].min().eq(0.0).all(),
    "numeric_max_is_one": df_reloaded[numeric_columns].max().eq(1.0).all(),
    "zipcode_preserved": df_reloaded["zipcode"].equals(df_original["zipcode"]),
    "city_preserved": df_reloaded["city"].equals(df_original["city"]),
    "raw_input_unchanged_in_memory": int(df_original.isna().sum().sum()) == 10,
}
validation_report = pd.Series(validation_checks, name="passed").rename_axis("check").to_frame()
display(validation_report)
assert validation_report["passed"].all()

,passed
check,
expected_shape_7_by_5,True
expected_columns,True
no_missing_values,True
no_duplicate_rows,True
numeric_min_is_zero,True
numeric_max_is_one,True
zipcode_preserved,True
city_preserved,True
raw_input_unchanged_in_memory,True


## 8. Reflection and trade-offs

- Dropping `extra_data` avoids inventing five of seven values, but it could discard a useful signal if the two observed values were especially informative. A real project would investigate why the field is missing before removal.
- Median imputation retains every row and is robust to outliers, but repeated median values shrink variance and may bias correlations. Model-aware or grouped imputation could be compared later.
- Min-max normalization improves scale comparability but depends on the observed range and can be distorted by future outliers. Z-score normalization is implemented as an alternative but was not selected for this bounded demonstration.
- All statistics here use the complete instructional sample. For predictive work, medians and scaling parameters must be learned on the training period only to prevent leakage.
- This policy must not be transferred mechanically to SPY price gaps. Missing trading observations may reflect exchange calendars, outages, or provider errors; full-history median prices would destroy time-series meaning.

In [11]:
print("Original shape -> cleaned shape:", df_original.shape, "->", df_cleaned.shape)
print("Original missing cells -> cleaned missing cells:", 10, "->", 0)
print("Removed columns:", dropped_columns)
print("Stage 06 preprocessing and artifact checks passed.")

Original shape -> cleaned shape: (7, 6) -> (7, 5)
Original missing cells -> cleaned missing cells: 10 -> 0
Removed columns: ['extra_data']
Stage 06 preprocessing and artifact checks passed.
